<a href="https://colab.research.google.com/github/asadsalamasad7-source/CodeAlpha_ProjectName/blob/main/customer_churn_litert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [29]:
import numpy as np
import pandas as pd

# Generate a realistic synthetic customer churn dataset
np.random.seed(42)
n_samples = 1000  # Increased sample size for better training stability

age = np.random.randint(18, 70, size=n_samples)
tenure = np.random.randint(1, 72, size=n_samples)
monthly_charges = np.random.uniform(20.0, 120.0, size=n_samples)
total_charges = tenure * monthly_charges + np.random.normal(0, 50, size=n_samples)
total_charges = np.clip(total_charges, 20.0, None)  # Prevent negative charges

# Create a churn probability based on tenure, age, and charges
# Lower tenure, higher monthly charges, and older age increase churn probability
churn_logit = -0.05 * tenure + 0.02 * monthly_charges + 0.01 * age - 1.5
churn_prob = 1 / (1 + np.exp(-churn_logit))
churn = np.random.binomial(1, churn_prob)

synthetic_data = pd.DataFrame({
    'CustomerID': range(1000, 1000 + n_samples),
    'Age': age,
    'Tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'Churn': churn
})

# Save it to the expected filename
synthetic_data.to_csv('customer_churn.csv', index=False)

# Read the newly created dataset
data = pd.read_csv('customer_churn.csv')
display(data.head())

,CustomerID,Age,Tenure,MonthlyCharges,TotalCharges,Churn
0,1000,56,15,55.162269,778.851194,1
1,1001,69,64,30.142083,1860.112394,0
2,1002,46,28,68.587176,1889.105064,0
3,1003,32,39,45.677657,1824.548288,0
4,1004,60,57,48.487290,2811.431793,0


In [30]:
print(data.head())
print(data.info())

   CustomerID  Age  Tenure  MonthlyCharges  TotalCharges  Churn
0        1000   56      15       55.162269    778.851194      1
1        1001   69      64       30.142083   1860.112394      0
2        1002   46      28       68.587176   1889.105064      0
3        1003   32      39       45.677657   1824.548288      0
4        1004   60      57       48.487290   2811.431793      0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      1000 non-null   int64  
 1   Age             1000 non-null   int64  
 2   Tenure          1000 non-null   int64  
 3   MonthlyCharges  1000 non-null   float64
 4   TotalCharges    1000 non-null   float64
 5   Churn           1000 non-null   int64  
dtypes: float64(2), int64(4)
memory usage: 47.0 KB
None


In [31]:
data = data.drop(columns=['CustomerID']) #Simplify the dataset
data = data.dropna()  # Simple example of dropping missing values

In [32]:
data = pd.get_dummies(data, drop_first=True)

In [33]:
from sklearn.preprocessing import StandardScaler

X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features to help the Neural Network converge faster and more accurately
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [34]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [35]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Use early stopping to prevent overfitting and more epochs for convergence
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5437 - loss: 0.7393 - val_accuracy: 0.7150 - val_loss: 0.6114
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6125 - loss: 0.6650 - val_accuracy: 0.6800 - val_loss: 0.6131
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6737 - loss: 0.6136 - val_accuracy: 0.6800 - val_loss: 0.6076
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6787 - loss: 0.5876 - val_accuracy: 0.6950 - val_loss: 0.5950
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7163 - loss: 0.5687 - val_accuracy: 0.6950 - val_loss: 0.5817
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7175 - loss: 0.5394 - val_accuracy: 0.7250 - val_loss: 0.5676
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7538 - loss: 0.5196 - val_accuracy: 0.7700 - val_loss: 0.5536
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7550 - loss: 0.5131 - val_accuracy: 0.7550 - 

In [36]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f'Test accuracy: {test_acc}')

7/7 - 0s - 8ms/step - accuracy: 0.7700 - loss: 0.4876
Test accuracy: 0.7699999809265137


In [37]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

Saved artifact at '/tmp/tmpuevl88tn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='keras_tensor_17')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132731614053968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614052624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614052816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614052432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614057616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614053392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614057808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614067600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614053200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614057232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132731614058192: Tens

In [39]:
import numpy as np
import tensorflow as tf

# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run prediction on the test set using TFLite interpreter
tflite_preds = []
for i in range(len(X_test)):
    input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    tflite_preds.append(output_data[0][0])

tflite_preds = np.array(tflite_preds)
tflite_binary_preds = (tflite_preds > 0.5).astype(int)

# Calculate TFLite test accuracy
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
tflite_acc = accuracy_score(y_test, tflite_binary_preds)

print("=== TFLite Model Evaluation ===")
print(f"TFLite Test Accuracy: {tflite_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, tflite_binary_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test, tflite_binary_preds))

=== TFLite Model Evaluation ===
TFLite Test Accuracy: 0.7700

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.92      0.86       150
           1       0.57      0.32      0.41        50

    accuracy                           0.77       200
   macro avg       0.69      0.62      0.63       200
weighted avg       0.74      0.77      0.75       200

Confusion Matrix:
[[138  12]
 [ 34  16]]


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [38]:
model.save('churn_model.h5')